[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnmarktaylor91/torchlens/blob/main/examples/methods/jacobian_lens_mamba_colab.ipynb)

# The Jacobian lens on Mamba-130m

**TorchLens fits the Jacobian lens (Gurnee et al., 2026) to Mamba-130m, a state-space model with no attention that the paper's reference code cannot fit, and reads what each of its 24 layers points to.**

In this notebook:

1. **Fit** the lens: one TorchLens trace per sentence and batched `torch.autograd.grad` give 23 Jacobian matrices.
2. **Read** every layer on four prompts, through the Jacobian lens and through the plain logit lens.
3. **Spot-check** the fitted Jacobian rows against exact scalar autograd.

**Runtime > Run all.** The model download is about 500 MB. The notebook runs on the CPU, even on a GPU runtime (a GPU path has not been tested); on a two-core CPU the whole notebook took between two and eight minutes in our runs.

## What the Jacobian lens is

Each block of a language model passes a residual vector per token to the next block. The Jacobian lens scores vocabulary tokens associated with a block's residual vector: it carries the vector into the final block's basis with one fixed matrix per block, the corpus-average Jacobian of the final block's output with respect to that block's output, and then applies the model's own final norm and unembedding. The logit lens is the special case where every matrix is the identity.

$$\mathrm{lens}_l(h) = \mathrm{unembed}\big(\mathrm{norm}_f(J_l\,h)\big), \qquad J_l = \mathbb{E}\left[\frac{\partial h_{L}}{\partial h_l}\right]$$

The paper's reference implementation adapts Hugging Face models through a fixed set of recognized model layouts, and Mamba is outside them, so the reference code cannot fit this model. TorchLens needs no registered adapter for Mamba: one trace per sentence captures the named blocks, and ordinary PyTorch computes the Jacobians.

Paper: Gurnee et al., "Verbalizable Representations Form a Global Workspace in Language Models", Transformer Circuits, July 2026, https://transformer-circuits.pub/2026/workspace/index.html

## Install

This installs TorchLens, transformers and matplotlib only when one of them is missing or older than the notebook needs; Colab already has PyTorch.

In [ ]:
from importlib.metadata import PackageNotFoundError, version


def at_least(package, floor):
    try:
        have = tuple(int(part) for part in version(package).split(".")[: len(floor)])
    except (PackageNotFoundError, ValueError):
        return False
    return have >= floor


if not (
    at_least("torchlens", (3, 0, 2))
    and at_least("transformers", (5, 18))
    and at_least("matplotlib", (3, 5))
):
    %pip install -q "torchlens>=3.0.2" "transformers>=5.18" "matplotlib>=3.5"

## Load Mamba-130m

The checkpoint is pinned to one revision, and the model runs on the CPU.

In [ ]:
import functools
import time

import torch
from transformers import AutoTokenizer, MambaForCausalLM, logging

import torchlens as tl

logging.set_verbosity_error()  # Mamba on CPU takes the plain PyTorch path, and transformers says so at length
device = "cpu"  # the tested path; the notebook runs on the CPU even on a GPU runtime

MODEL_ID = "state-spaces/mamba-130m-hf"
REVISION = "1e76775f628fbf1350fbe4dbb3d971ba64af25a1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
model = (
    MambaForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, use_cache=False).eval().to(device)
)

layers = [f"backbone.layers.{i}" for i in range(len(model.backbone.layers))]
final_norm, unembed = model.backbone.norm_f, model.lm_head
d_model = model.config.hidden_size
print(
    f"{len(layers)} layers, d_model = {d_model}, TorchLens {tl.__version__}, torch {torch.__version__}, {device}"
)

## Fit the lens with TorchLens

`residual_states` is the TorchLens part: one `tl.trace` returns every block's live output, and `prompt_jacobians` differentiates the final block's output with respect to each of them. This is the slow cell; it prints its own time.

In [ ]:
CORPUS = [
    "The cat sat on the mat while rain fell outside.",
    "A little boat crossed the lake before the sun went down.",
]
SKIP_FIRST = 1  # positions skipped at the start of each sentence
CHUNK = 4  # cotangents per backward pass; on Mamba's CPU path larger blocks gain nothing


def residual_states(ids):
    trace = tl.trace(model, ids, capture=tl.options.CaptureOptions(save_mode="view"))
    return [trace[name].out for name in layers]


@functools.cache
def prompt_jacobians(text):
    # One sentence's Jacobian rows for every earlier block, averaged over its source positions.
    states = residual_states(tokenizer(text, return_tensors="pt").input_ids.to(device))
    target = states[-1][0, SKIP_FIRST:-1].sum(0)
    rows = [[] for _ in layers[:-1]]
    for cotangents in torch.eye(d_model, device=device).split(CHUNK):
        grads = torch.autograd.grad(
            target, states[:-1], grad_outputs=cotangents, is_grads_batched=True, retain_graph=True
        )
        for row_list, grad in zip(rows, grads):
            row_list.append(grad[:, 0, SKIP_FIRST:-1].mean(1).detach())
    return [torch.cat(row_list) for row_list in rows]


def fit_jacobian_lens(texts):
    # Corpus-average Jacobian of the final block's output with respect to each earlier block's.
    per_prompt = [prompt_jacobians(text) for text in texts]
    return [sum(rows[i] for rows in per_prompt) / len(texts) for i in range(len(layers) - 1)]


started = time.perf_counter()
J = fit_jacobian_lens(CORPUS)
print(
    f"fitted {len(J)} matrices of shape {tuple(J[0].shape)} in {time.perf_counter() - started:.0f} s"
)

**How the fit works.** `tl.trace(model, ids, capture=CaptureOptions(save_mode="view"))` runs one forward pass and keeps every block's live output on the autograd graph (the default mode saves copies, which have no path back to the model's computation). `trace["backbone.layers.7"].out` is then the actual residual tensor that block 8 consumed, so `torch.autograd.grad` can differentiate the final block's output with respect to it.

The estimator is the one in the TorchLens gallery entry. For each sentence, the final block's output is summed over positions 1 to T-2 (the first and the last token are left out), and a block of cotangents (output directions, `CHUNK` at a time) gives that many Jacobian rows for every earlier block in one backward pass. The rows are averaged over the same source positions, then over sentences with equal weight. Nothing here is specific to Mamba beyond the module names.

## What each layer reads out

The readout is the paper's: transport the last position's residual with $J_l$, then apply Mamba's final norm and unembedding. The next cell holds the readout, table and figure code (collapsed in Colab; double-click to open it).

In [ ]:
# @title Readout, table and figure helpers (plain PyTorch and matplotlib)
import io
import textwrap

import matplotlib.pyplot as plt
from IPython.display import Image, display
from PIL import Image as PILImage


def lens_probs(h, matrix=None):
    with torch.no_grad():
        return torch.softmax(unembed(final_norm(h if matrix is None else h @ matrix.T)), -1)


def read_prompt(text):
    ids = tokenizer(text, return_tensors="pt").input_ids.to(device)
    with torch.no_grad():
        states = [h[0, -1].detach() for h in residual_states(ids)]
    answer = int(lens_probs(states[-1]).argmax())
    out = {"answer": tokenizer.decode(answer)}
    for name, matrices in (("Jacobian lens", J + [None]), ("logit lens", [None] * len(layers))):
        probs = [lens_probs(h, m) for h, m in zip(states, matrices)]
        out[name] = [
            (tokenizer.decode(int(p.argmax())), float(p.max()), float(p[answer])) for p in probs
        ]
    return out


def show_png(path, width):
    # Display a palette-quantized copy so the stored notebook stays small; the file on disk is full colour.
    buffer = io.BytesIO()
    PILImage.open(path).quantize(colors=96).save(buffer, "PNG", optimize=True)
    display(Image(data=buffer.getvalue(), width=width))


INK, MUTED, RULE, PLAIN = "#1f1f1e", "#6b6b66", "#d9d8d2", "#f7f6f2"
HIT = "#0072b2"  # blue: the cell's top token is the prompt's answer word
LENSES = ("Jacobian lens", "logit lens")


def show(token, limit=12):
    text = token.strip()
    if not text:
        return "(space)"
    if "\ufffd" in text:
        return "(bytes)"  # part of a multi-byte character; the tokenizer decodes it as \ufffd
    return text if len(text) <= limit else text[: limit - 1] + "…"


def is_answer(token, answer):
    return token.strip() == answer


def draw_cell(ax, x, y, token, answer, font, limit=12):
    hit = is_answer(token, answer)
    ax.add_patch(
        plt.Rectangle(
            (x, y), 1, 1, facecolor=HIT if hit else PLAIN, edgecolor="white", linewidth=1.2
        )
    )
    ax.text(
        x + 0.5,
        y + 0.5,
        show(token, limit),
        ha="center",
        va="center",
        fontsize=font,
        family="monospace",
        color="white" if hit else INK,
        fontweight="bold" if hit else "normal",
    )


def bare(ax, n_cols, n_rows):
    ax.set_xlim(0, n_cols)
    ax.set_ylim(n_rows, 0)
    ax.set_xticks([])
    ax.set_yticks([])
    for side in ax.spines.values():
        side.set_visible(False)


def draw_lenses(
    readouts, prompts, answers, layers_shown, title, subtitle, footer, per_row=2, path=None
):
    # Main figure: one panel per prompt, layers down the side, the two lenses side by side in
    # each panel. per_row=2 is drawn for a phone screen (390 CSS px wide at 3x).
    width, font, small, row = 1.95 * per_row, 9.5, 9.5, 0.205
    margin, label_w, gutter = 0.1, 0.36, 0.14
    title_lines = textwrap.wrap(title, 18 * per_row)
    sub_lines = textwrap.wrap(subtitle, 25 * per_row)
    foot_lines = textwrap.wrap(footer, 25 * per_row)
    top = 0.1 + 0.215 * len(title_lines) + 0.165 * len(sub_lines) + 0.12
    head, col_head = 0.66, 0.19
    block = head + col_head + row * len(layers_shown)
    bands = -(-len(prompts) // per_row)
    height = top + bands * block + (bands - 1) * 0.2 + 0.2 + 0.165 * len(foot_lines) + 0.12
    panel_w = (width - 2 * margin - label_w - gutter * (per_row - 1)) / per_row
    fig = plt.figure(figsize=(width, height), dpi=300)
    y = height - 0.1
    for line in title_lines:
        fig.text(
            margin / width, y / height, line, fontsize=12, fontweight="bold", va="top", color=INK
        )
        y -= 0.215
    for line in sub_lines:
        fig.text(margin / width, y / height, line, fontsize=small, va="top", color=MUTED)
        y -= 0.165
    for k, text in enumerate(prompts):
        col, band = k % per_row, k // per_row
        x0 = margin + label_w + col * (panel_w + gutter)
        y_top = height - top - band * (block + 0.2)
        grid_h = row * len(layers_shown)
        ax = fig.add_axes([x0 / width, (y_top - block) / height, panel_w / width, grid_h / height])
        bare(ax, 2, len(layers_shown))
        prompt_lines = textwrap.wrap(f"“{text}”", 25)
        for j, line in enumerate(prompt_lines):
            fig.text(
                x0 / width, (y_top - 0.155 * j) / height, line, fontsize=small, va="top", color=INK
            )
        y_line = y_top - 0.155 * len(prompt_lines)
        fig.text(
            x0 / width,
            y_line / height,
            f"answer: {answers[text]}",
            fontsize=small,
            va="top",
            color=HIT,
        )
        fig.text(
            x0 / width,
            (y_line - 0.155) / height,
            f"model says: {show(readouts[text]['answer'])}",
            fontsize=small,
            va="top",
            color=MUTED,
        )
        for c, name in enumerate(("Jacobian", "logit")):
            ax.text(
                c + 0.5,
                -0.25,
                name,
                ha="center",
                va="bottom",
                fontsize=small,
                color=INK if c == 0 else MUTED,
                fontweight="bold" if c == 0 else "normal",
            )
        for r, layer in enumerate(layers_shown):
            for c, name in enumerate(LENSES):
                draw_cell(ax, c, r, readouts[text][name][layer][0], answers[text], font, 9)
            if col == 0:
                ax.text(
                    -0.08, r + 0.5, str(layer), ha="right", va="center", fontsize=small, color=MUTED
                )
        if col == 0:
            ax.text(-0.08, -0.25, "layer", ha="right", va="bottom", fontsize=small, color=MUTED)
    y = height - top - bands * block - (bands - 1) * 0.2 - 0.2 - 0.04
    for line in foot_lines:
        fig.text(margin / width, y / height, line, fontsize=small, va="top", color=MUTED)
        y -= 0.165
    if path:
        fig.savefig(path, dpi=300, facecolor="white")
    return fig


def draw_wide(readouts, prompts, answers, layers_shown, title, notes, path=None):
    # Detail figure: 1920 x 1080, the two lenses side by side, every layer.
    font, step, title_y = 15, 0.026, 0.955
    notes = [line for note in notes for line in textwrap.wrap(note, 165)]
    fig, axes = plt.subplots(1, 2, figsize=(19.2, 10.8), dpi=100)
    fig.subplots_adjust(
        left=0.075,
        right=0.985,
        top=title_y - step * (len(notes) + 1) - 0.16,
        bottom=0.02,
        wspace=0.04,
    )
    for ax, name in zip(axes, LENSES):
        bare(ax, len(prompts), len(layers_shown))
        for c, text in enumerate(prompts):
            for r, layer in enumerate(layers_shown):
                draw_cell(ax, c, r, readouts[text][name][layer][0], answers[text], font, 14)
            lines = [
                (f"model says: {show(readouts[text]['answer'])}", MUTED),
                (f"answer: {answers[text]}", HIT),
                (textwrap.fill(f"“{text}”", 18), INK),
            ]
            for k, (line, color) in enumerate(lines):
                ax.annotate(
                    line,
                    (c + 0.5, -0.4),
                    xytext=(0, 19 * k + (4 if k == 2 else 0)),
                    textcoords="offset points",
                    ha="center",
                    va="bottom",
                    fontsize=font,
                    color=color,
                    linespacing=1.1,
                    annotation_clip=False,
                )
        ax.set_title(
            name,
            loc="left",
            fontsize=font + 3,
            pad=124,
            fontweight="bold" if name == "Jacobian lens" else "normal",
            color=INK if name == "Jacobian lens" else MUTED,
        )
    for r, layer in enumerate(layers_shown):
        axes[0].text(
            -0.06, r + 0.5, f"layer {layer}", ha="right", va="center", fontsize=font, color=MUTED
        )
    fig.text(0.012, title_y, title, fontsize=21, fontweight="bold", va="bottom", color=INK)
    for k, line in enumerate(notes):
        fig.text(0.012, title_y - step * (k + 1), line, fontsize=font, va="bottom", color=INK)
    if path:
        fig.savefig(path, dpi=100, facecolor="white")
    return fig


def print_table(readouts, prompts, layers_shown):
    # Each lens's top token and the probability it gives that token, one block per prompt.
    for text in prompts:
        print(f"{text} ...   (model says: {show(readouts[text]['answer'])})")
        print(f"  {'layer':>5}   {'Jacobian lens':<20}{'logit lens':<20}")
        for layer in layers_shown:
            cells = []
            for name in LENSES:
                token, p_top, _ = readouts[text][name][layer]
                cells.append(f"{show(token):<13}{p_top:>5.2f}  ")
            print(f"  {layer:>5}   {cells[0]}{cells[1]}")
        print()

Each cell of the figure is a lens's top token at that layer, for the last token of the prompt. Blue marks the intended answer (Paris, cold, France, Jupiter), which is not always what the model says. Layer 23 is the model's own output, so both lenses agree there by construction.

In [ ]:
PROMPTS = [
    "The Eiffel Tower is in the city of",
    "The opposite of hot is",
    "Paris is the capital of",
    "The largest planet in the solar system is",
]
ANSWERS = dict(zip(PROMPTS, ["Paris", "cold", "France", "Jupiter"]))
readouts = {text: read_prompt(text) for text in PROMPTS}

draw_lenses(
    readouts,
    PROMPTS,
    ANSWERS,
    range(17, 24),
    title="Mamba-130m: the Jacobian lens first names the answer at layer 19 or 20, the logit lens at 22 or not at all",
    subtitle="Four prompts chosen from twenty: a demonstration of the method, not an accuracy measurement. Each cell is the word a lens ranks first at that layer, read at the prompt's last token. Blue: the intended answer.",
    footer="Jacobian lens: the layer's hidden state times a matrix fitted on two other sentences (CORPUS), then the model's own output head; the logit lens skips the matrix. Layers 17 to 23 of 24 shown; below 17 neither lens names the answer. Layer 23 is the model's output. Tokens can be word pieces (openhagen); (bytes) is part of a multi-byte character.",
    path="mamba_lens_late.png",
)
plt.close("all")
show_png("mamba_lens_late.png", 420)

On these four prompts, chosen from twenty we tried, the Jacobian lens reads Paris and cold from layer 20, where the logit lens still gives commas and function words; the logit lens reaches those two words at layer 22. On the two prompts where the model itself answers " the", the Jacobian lens reads " France" from layer 19, and " Jupiter", " Saturn", " Jupiter" at layers 20 to 22. These are the top tokens of a two-sentence fit on four selected prompts, not a measurement of the lens's accuracy.

The same readouts as a table, with the probability each lens gives its own top token:

In [ ]:
print_table(readouts, PROMPTS, range(17, 24))

### All 24 layers

Below layer 19 the Jacobian-lens readouts are mostly fragments: byte pieces (shown as "(bytes)": part of a multi-byte character, which the tokenizer decodes as "��"), symbols and subword pieces such as "openhagen". A larger, representative corpus may change them; in our runs eight sentences did not.

In [ ]:
TITLE_ALL = (
    "All 24 layers: answer first at layer 19 or 20 (Jacobian lens), at 22 or never (logit lens)"
)
NOTES_ALL = [
    "Mamba-130m, four prompts chosen from twenty: a demonstration of the method, not an accuracy measurement. J is fitted on two other sentences; below layer 19 the Jacobian lens reads mostly fragments.",
    "Jacobian lens: the last token's hidden state times J, then the model's final norm and unembedding; the logit lens omits J. Layer n is the output of block n; layer 23 is the model's output.",
    "Cell text: the token the lens gives the highest probability; blue: the intended answer named under each prompt. Leading spaces dropped; (space): a whitespace token; (bytes): part of a multi-byte character; []$ and }{~ are literal symbol tokens; openhagen and similar are word pieces.",
]
draw_wide(
    readouts,
    PROMPTS,
    ANSWERS,
    range(len(layers)),
    TITLE_ALL,
    NOTES_ALL,
    path="mamba_lens_all_layers.png",
)
plt.close("all")
show_png("mamba_lens_all_layers.png", 960)

## Spot-check against exact autograd

Nothing here uses TorchLens: plain forward hooks catch the block outputs, and `torch.autograd.grad` on one scalar output at a time gives exact Jacobian rows, reduced the same way as the fit. The cell compares the first 64 of the 768 rows at three blocks for one corpus sentence with the TorchLens rows for that sentence (cached during the fit).

In [ ]:
CHECK_LAYERS, CHECK_ROWS = (0, 11, 22), 64
text = CORPUS[0]
ids = tokenizer(text, return_tensors="pt").input_ids.to(device)

seen = {}
handles = [
    model.backbone.layers[i].register_forward_hook(
        lambda module, args, out, i=i: seen.__setitem__(
            i, out[0] if isinstance(out, tuple) else out
        )
    )
    for i in range(len(layers))
]
model(ids, use_cache=False)
for handle in handles:
    handle.remove()

target = seen[len(layers) - 1][0, SKIP_FIRST:-1].sum(0)
sources = [seen[i] for i in CHECK_LAYERS]
started = time.perf_counter()
exact = {i: [] for i in CHECK_LAYERS}
for value in target[:CHECK_ROWS]:
    for i, grad in zip(CHECK_LAYERS, torch.autograd.grad(value, sources, retain_graph=True)):
        exact[i].append(grad[0, SKIP_FIRST:-1].mean(0).detach())
print(
    f"{CHECK_ROWS} exact rows at {len(CHECK_LAYERS)} blocks in {time.perf_counter() - started:.0f} s"
)

captured = prompt_jacobians(text)  # the TorchLens rows for the same sentence, cached from the fit
for i in CHECK_LAYERS:
    reference = torch.stack(exact[i])
    error = float((captured[i][:CHECK_ROWS] - reference).norm() / reference.norm())
    print(f"block {i:>2}: relative Frobenius error {error:.1e}")

Scalar reverse mode through Mamba's sequential scan is slow, which is why the check samples rows. On an x86 CPU the batched and scalar passes differ by float32 round-off, about 1e-7; on Apple silicon in our run they agreed exactly. The full comparison, every row of all 23 matrices of the two-sentence fit against the same scalar oracle, is the TorchLens gallery entry linked below, where the largest relative error was 4.8e-7 (layer 0) and the smallest 4.8e-8 (layer 22). The two checks have different scopes, so their numbers are not expected to match.

## Notes

- The corpus is two sentences, so the matrices are a demonstration of the mechanics, not a fit of the paper's dataset. Add sentences to `CORPUS` to try a larger fit; the cost grows linearly, about 768 scalar-equivalent backward passes per sentence on a CPU.
- `save_mode="view"` keeps live tensors. A model that overwrote its residual stream in place would change the saved state; `save_mode="reference"` raises on a read after such a write.
- The fit runs with gradients on; only the readouts and the figures run under `torch.no_grad()`.

## Links

- TorchLens: https://github.com/johnmarktaylor91/torchlens
- The checked gallery entry, with GPT-2 small against the paper's reference code and Mamba-130m against exact autograd over all 23 layers: https://github.com/johnmarktaylor91/torchlens/blob/main/docs/gallery/jacobian_lens.md
- A step-by-step GPT-2 version of this recipe: https://github.com/johnmarktaylor91/torchlens/blob/main/examples/methods/jacobian_lens_recipe.ipynb